# LeetCode 508 — Most Frequent Subtree Sum bằng Python

Link đề: https://leetcode.com/problems/most-frequent-subtree-sum/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Hash Table, Binary Tree, Tree DP

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def findFrequentTreeSum(self, root: Optional[TreeNode]) -> List[int]:
        ...
```

Ghi chú: bài cho **nhiều đáp án đúng** (khi có nhiều tổng cùng đứng đầu về tần suất thì trả về tất cả theo **thứ tự bất kỳ**), nên trong notebook ta so sánh đáp án **sau khi sắp xếp** hoặc bằng hàm kiểm tra tính hợp lệ. Notebook tự dựng `TreeNode` và hàm `build_tree` dựng cây từ list kiểu LeetCode (`[5,2,-3]`). Ví dụ số 3 (cây một nút) và các cây trong phần tự kiểm là do notebook tự đặt, đáp án đã kiểm chứng bằng code; nếu cần hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **tổng cây con (subtree sum)** của một nút là tổng giá trị mọi nút trong cây con gốc tại nút đó (kể cả chính nó); cần tìm tổng xuất hiện **nhiều lần nhất** trong số `n` tổng của `n` nút, và trả về **tất cả** tổng hòa nhau ở đầu bảng.
2. Nắm các công cụ Python: định nghĩa `class TreeNode`, dựng cây từ list có `None`, đệ quy hậu tự (post-order), `collections.Counter`, `max(...)` trên các giá trị, hàm lồng nhau và `nonlocal`, ngăn xếp tường minh, `itertools.groupby`.
3. Đi từ **mỗi nút tự tính lại tổng cây con** (`O(n²)` ở trường hợp xấu) → **một lượt DFS hậu tự tính tổng và đếm cùng lúc** (`O(n)`) → **bản không đệ quy** → **duyệt theo tầng rồi cộng dồn ngược lên cha** → **đếm bằng sắp xếp** thay vì bảng băm.
4. Hiểu mấu chốt: `sum(node) = node.val + sum(left) + sum(right)`, nên phải biết tổng của **hai con trước** khi tính cho cha → **post-order**; mỗi nút chỉ cần tính **một lần** nếu ta *trả tổng về cho cha*.
5. Tránh các lỗi kinh điển: quên cộng `node.val`, chỉ đếm tổng ở gốc, chỉ trả về **một** đáp án khi bị hòa, để trạng thái đếm lưu lại giữa các lần gọi, và nhầm "tần suất lớn nhất" với "tổng lớn nhất".

## 2. Đề bài & ví dụ

Cho nút gốc `root` của một cây nhị phân. Hãy trả về **tổng cây con xuất hiện nhiều lần nhất**. Nếu có nhiều giá trị cùng xuất hiện nhiều nhất thì trả về **tất cả** các giá trị đó, thứ tự bất kỳ.

**Tổng cây con** của một nút là tổng của mọi giá trị nút trong cây con có gốc là nút đó (bao gồm chính nút đó).

**Ví dụ 1**

- Input: `root = [5,2,-3]`
- Output: `[2,-3,4]`
- Giải thích: ba tổng là `2` (nút 2), `-3` (nút -3) và `5 + 2 + (-3) = 4` (gốc). Mỗi tổng xuất hiện đúng 1 lần nên đều là đáp án.

**Ví dụ 2**

- Input: `root = [5,2,-5]`
- Output: `[2]`
- Giải thích: các tổng là `2`, `-5` và `5 + 2 - 5 = 2`. Tổng `2` xuất hiện **2 lần**, nhiều nhất.

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `root = [1]`
- Output: `[1]`

**Ràng buộc:**

- Số nút của cây nằm trong khoảng `[1, 10^4]`.
- `-10^5 <= Node.val <= 10^5`

Chú ý: đề cho phép trả về theo **thứ tự bất kỳ**, nên `[2,-3,4]` và `[4,2,-3]` đều đúng.

In [1]:
from typing import Optional, List
from collections import Counter, deque, defaultdict
from itertools import groupby
import random
import sys

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([5, 2, -3], [-3, 2, 4]),     # đáp án đã sắp xếp
    ([5, 2, -5], [2]),
    ([1], [1]),
]
for vals, expected in examples:
    assert tree_to_list(build_tree(vals)) == vals
    print(vals, "-> kỳ vọng (sắp xếp)", expected)

[5, 2, -3] -> kỳ vọng (sắp xếp) [-3, 2, 4]
[5, 2, -5] -> kỳ vọng (sắp xếp) [2]
[1] -> kỳ vọng (sắp xếp) [1]


## 3. Kiến thức nền

### 3.1. Nút cây và cây nhị phân

Mỗi nút (`TreeNode`) có `val` (giá trị), `left`, `right` (nút con hoặc `None`). **Chỉ cần giữ nút gốc là truy cập được cả cây.** LeetCode định nghĩa sẵn `TreeNode` (khi nộp bài không cần viết lại), ở đây ta tự định nghĩa ở cell trên để chạy được tại máy.

In [2]:
leaf_a = TreeNode(2)
leaf_b = TreeNode(-3)
root = TreeNode(5, leaf_a, leaf_b)
print(root.val, root.left.val, root.right.val)
print("con trái của lá:", leaf_a.left)        # None: lá không có con

5 2 -3
con trái của lá: None


### 3.2. Dựng cây từ list kiểu LeetCode

Trong đề, cây được cho dạng `[5,2,-3]`, `[1,2,3,None,4]`... theo **thứ tự từng tầng, từ trái sang phải**, `None` là chỗ trống. Hàm `build_tree` dùng một hàng đợi: lấy nút đang chờ, gán con trái rồi con phải từ list, rồi đưa các con thật vào hàng đợi. `tree_to_list` làm ngược lại — ta dùng nó để kiểm tra việc dựng.

Hàm `show_levels` dưới đây in từng tầng để bạn "nhìn" được cây:

In [3]:
def show_levels(root):
    level, d = [root], 0
    while any(x is not None for x in level):
        print(f"tầng {d}:", [x.val if x else None for x in level])
        level = [c for x in level if x for c in (x.left, x.right)]
        d += 1

t = build_tree([5, 2, -3, 4, None, None, 1])
show_levels(t)
print(tree_to_list(t))

tầng 0: [5]
tầng 1: [2, -3]
tầng 2: [4, None, None, 1]
[5, 2, -3, 4, None, None, 1]


### 3.3. Đệ quy hậu tự (post-order)

Với cây, ta hay viết hàm đệ quy `dfs(node)`:

1. Nếu `node is None` → trả về giá trị "rỗng" (ở đây là `0`).
2. Gọi `dfs` cho **con trái** và **con phải** trước.
3. Dùng kết quả của hai con để tính kết quả cho `node`.

Thứ tự "con trước, cha sau" gọi là **hậu tự (post-order)**. Nó phù hợp mỗi khi đáp án của cha phụ thuộc đáp án của con. Ví dụ đơn giản: đếm số nút của cây.

In [4]:
def count_nodes(node):
    if node is None:
        return 0
    return count_nodes(node.left) + count_nodes(node.right) + 1

print(count_nodes(build_tree([5, 2, -3, 4, None, None, 1])))   # 5
print(count_nodes(None))                                       # 0

5
0


### 3.4. `Counter` để đếm tần suất

`collections.Counter` là một `dict` chuyên đếm: `cnt[x] += 1` luôn chạy được kể cả khi `x` chưa có (mặc định `0`). `cnt.values()` là các tần suất; `max(cnt.values())` là tần suất lớn nhất; `cnt.most_common()` liệt kê theo tần suất giảm dần.

In [5]:
cnt = Counter()
for x in [2, 4, 2, -3, 2, 4]:
    cnt[x] += 1
print(cnt)
print("tần suất lớn nhất:", max(cnt.values()))
print("các khóa đạt tần suất đó:", [k for k, v in cnt.items() if v == max(cnt.values())])
print(cnt.most_common(2))
print(Counter([1, 1, 2, 2, 3]))       # bị hòa: 1 và 2 cùng xuất hiện 2 lần

Counter({2: 3, 4: 2, -3: 1})
tần suất lớn nhất: 3
các khóa đạt tần suất đó: [2]
[(2, 3), (4, 2)]
Counter({1: 2, 2: 2, 3: 1})


### 3.5. Hàm lồng nhau và `nonlocal`

Hàm `dfs` hay được định nghĩa **bên trong** hàm chính để dùng chung biến (như `cnt`). Nếu chỉ **đọc / gọi phương thức** trên biến ngoài (ví dụ `cnt[s] += 1` hay `res.append(...)`) thì không cần gì thêm. Nhưng nếu muốn **gán lại** một biến số (`best = ...`), phải khai báo `nonlocal best`.

In [6]:
def demo():
    total = 0
    items = []
    def add(x):
        nonlocal total          # vì có phép gán total = ...
        total += x
        items.append(x)         # append không cần nonlocal
    for v in [3, 4, 5]:
        add(v)
    return total, items

print(demo())

(12, [3, 4, 5])


### 3.6. "Nhiều đáp án đúng" và cách so sánh

Với bài này, hai list `[2,-3,4]` và `[4,2,-3]` đều đúng. Khi viết test, ta **sắp xếp** cả hai rồi mới so sánh, hoặc viết hàm kiểm tra riêng "đáp án có đúng là tập các tổng có tần suất lớn nhất không". Nhớ rằng `sorted(a) == sorted(b)` là cách so sánh **tập hợp có lặp**; bài này đáp án không có phần tử lặp nên rất hợp.

In [7]:
a = [2, -3, 4]
b = [4, 2, -3]
print(a == b, sorted(a) == sorted(b))

False True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét cây `[5, 2, -3]`:

```
    5
   / \
  2   -3
```

Mỗi nút có một "tổng cây con":

- Nút `2` là lá: tổng cây con = `2`.
- Nút `-3` là lá: tổng cây con = `-3`.
- Nút `5` (gốc): `5 + 2 + (-3) = 4`.

Ta thu được bảng `n = 3` tổng: `{2, -3, 4}`, mỗi tổng 1 lần → trả về cả ba.

Giờ xét `[5, 2, -5]`: tổng là `2`, `-5`, và gốc `5 + 2 + (-5) = 2`. Bảng đếm: `{2: 2, -5: 1}` → tần suất lớn nhất là `2`, chỉ có tổng `2` đạt → đáp án `[2]`.

Quy trình tự nhiên: **(1)** tính tổng cây con của từng nút, **(2)** đếm tần suất, **(3)** lọc ra các tổng có tần suất lớn nhất. Vấn đề cần suy nghĩ là: **tính tổng cây con của từng nút** sao cho rẻ.

Quan sát quan trọng: tổng cây con của cha **dùng lại** tổng của hai con:

```
sum(node) = node.val + sum(node.left) + sum(node.right)
```

Chạy tay trên cây `[1, 2, 3, 4]` (nút 4 là con trái của 2):

- `sum(4) = 4`
- `sum(2) = 2 + sum(4) + 0 = 6`
- `sum(3) = 3`
- `sum(1) = 1 + 6 + 3 = 10`

In [8]:
t = build_tree([1, 2, 3, 4])
show_levels(t)
print("sum(4)=4, sum(2)=2+4=6, sum(3)=3, sum(1)=1+6+3=10")

tầng 0: [1]
tầng 1: [2, 3]
tầng 2: [4, None, None, None]
sum(4)=4, sum(2)=2+4=6, sum(3)=3, sum(1)=1+6+3=10


## 5. Cách 1 — Brute Force / baseline: mỗi nút tự tính lại tổng cây con

**Ý tưởng:**

1. Duyệt qua **mọi nút** của cây (bằng một DFS đơn giản).
2. Với mỗi nút, chạy **một lượt duyệt riêng** qua cây con của nó để cộng các giá trị (`naive_subtree_sum`).
3. Đếm các tổng bằng `Counter`, rồi lọc những tổng có tần suất lớn nhất.

Đây đúng là cách "dịch nguyên văn đề bài" nên rất dễ đúng.

In [9]:
def naive_subtree_sum(node):
    """Cộng mọi giá trị trong cây con của node (một lượt duyệt riêng)."""
    if node is None:
        return 0
    return node.val + naive_subtree_sum(node.left) + naive_subtree_sum(node.right)


def freq_sum_brute(root):
    all_nodes = []
    stack = [root]
    while stack:
        node = stack.pop()
        if node is None:
            continue
        all_nodes.append(node)
        stack.append(node.left)
        stack.append(node.right)

    cnt = Counter()
    for node in all_nodes:
        cnt[naive_subtree_sum(node)] += 1        # tính lại từ đầu cho từng nút
    best = max(cnt.values())
    return [s for s, c in cnt.items() if c == best]

for vals, expected in examples:
    got = freq_sum_brute(build_tree(vals))
    print(vals, "->", got, "| kỳ vọng (sắp xếp)", expected)
    assert sorted(got) == expected

[5, 2, -3] -> [4, -3, 2] | kỳ vọng (sắp xếp) [-3, 2, 4]
[5, 2, -5] -> [2] | kỳ vọng (sắp xếp) [2]
[1] -> [1] | kỳ vọng (sắp xếp) [1]


**Độ phức tạp.** Mỗi nút kích hoạt một lượt duyệt qua cây con của nó. Với cây **lệch** (mỗi nút chỉ có một con, độ sâu `n`), nút ở độ sâu `d` có cây con `n - d` nút → tổng `n + (n-1) + ... + 1 = O(n²)`. Với `n = 10^4` là khoảng `5·10^7` phép — chậm với Python.

- **Time:** `O(n²)` trường hợp xấu (cây lệch), `O(n log n)` nếu cây cân bằng.
- **Space:** `O(n)` (danh sách nút + ngăn xếp/đệ quy).

**Nút thắt:** cây con của nút `2` ở trên bị duyệt **nhiều lần** (một lần khi tính cho `2`, một lần nữa khi tính cho cha của `2`). Ta đã có sẵn tổng của con khi tính cho cha, vì sao phải tính lại?

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** để tính `sum(node)` ta cần `sum(left)` và `sum(right)`; mà chính ta vừa tính chúng khi đệ quy xuống. Vậy hãy cho hàm đệ quy **trả về tổng cây con** của nút nó đang xử lý:

```
dfs(node):
    nếu node rỗng: trả về 0
    s = node.val + dfs(node.left) + dfs(node.right)
    cnt[s] += 1             # ghi nhận tổng của nút này
    trả về s                # để cha dùng lại
```

Mỗi nút được xử lý **đúng một lần**, nên tổng thời gian `O(n)`.

Sau khi duyệt xong, `cnt` chứa tần suất của **mọi** tổng cây con. Ta lấy `best = max(cnt.values())` rồi trả về mọi khóa có tần suất `best`. Việc xử lý "nhiều đáp án" tự nhiên được: ta duyệt qua toàn bộ `cnt` thay vì chỉ lấy một phần tử.

Đây là một ví dụ nhỏ của **Tree DP**: giá trị của nút cha được tính từ giá trị của các nút con, và ta giữ thêm một "sổ ghi chép" (`cnt`) cập nhật trong lúc tính.

## 7. Các cách tiếp theo

### Cách 2 — DFS hậu tự + `Counter` (cách chuẩn, `O(n)`)

**Giải thích từng đoạn:**

1. `cnt = Counter()` — sổ đếm tần suất các tổng cây con.
2. `dfs(node)`: gặp `None` trả `0`. Ngược lại tính `s = node.val + dfs(left) + dfs(right)`.
3. `cnt[s] += 1` rồi `return s`.
4. Sau `dfs(root)`: `best = max(cnt.values())`, trả `[s for s, c in cnt.items() if c == best]`.

In [10]:
def freq_sum_dfs(root):
    cnt = Counter()

    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        cnt[s] += 1
        return s

    dfs(root)
    best = max(cnt.values())
    return [s for s, c in cnt.items() if c == best]

for vals, expected in examples:
    got = freq_sum_dfs(build_tree(vals))
    print(vals, "->", got)
    assert sorted(got) == expected

[5, 2, -3] -> [2, -3, 4]
[5, 2, -5] -> [2]
[1] -> [1]


In [11]:
class Solution:
    def findFrequentTreeSum(self, root: Optional[TreeNode]) -> List[int]:
        cnt = Counter()

        def dfs(node: Optional[TreeNode]) -> int:
            if node is None:
                return 0
            s = node.val + dfs(node.left) + dfs(node.right)
            cnt[s] += 1
            return s

        dfs(root)
        best = max(cnt.values())
        return [s for s, c in cnt.items() if c == best]

print([sorted(Solution().findFrequentTreeSum(build_tree(v))) for v, _ in examples])

[[-3, 2, 4], [2], [1]]


**Độ phức tạp (cách 2):**

- **Time:** `O(n)` — mỗi nút một lần, mỗi lần `O(1)` cho bảng băm (trung bình); lọc cuối `O(số tổng khác nhau) ≤ O(n)`.
- **Space:** `O(n)` — bảng `cnt` tối đa `n` khóa, cộng ngăn xếp đệ quy `O(h)` với `h` là chiều cao cây (`h` có thể bằng `n` với cây lệch).

### Cách 3 — Hậu tự **không đệ quy** (ngăn xếp + từ điển tổng)

Cây lệch `10^4` nút có thể làm đệ quy Python sâu `10^4` tầng (giới hạn mặc định chỉ khoảng `1000`; LeetCode thường nới ra, nhưng ở máy cá nhân sẽ báo `RecursionError`). Dùng **ngăn xếp tường minh** để tránh.

**Giải thích:**

1. Ngăn xếp chứa cặp `(node, đã_xử_lý_con_chưa)`.
2. Lấy `(node, False)` ra: đẩy lại `(node, True)` rồi đẩy hai con (nếu có) với cờ `False`. Như vậy `node` sẽ được **lấy ra lần hai sau hai con** → đó là thời điểm hậu tự.
3. Lấy `(node, True)`: cả hai con đã có trong `sums` (dict từ **nút** → tổng cây con). Tính `s = node.val + sums.get(left, 0) + sums.get(right, 0)`, ghi `sums[node] = s`, `cnt[s] += 1`.

In [12]:
def freq_sum_iter(root):
    cnt = Counter()
    sums = {}                                   # nút -> tổng cây con
    stack = [(root, False)]
    while stack:
        node, done = stack.pop()
        if node is None:
            continue
        if not done:
            stack.append((node, True))          # lần sau gặp lại: con đã xong
            stack.append((node.right, False))
            stack.append((node.left, False))
        else:
            s = node.val + sums.get(node.left, 0) + sums.get(node.right, 0)
            sums[node] = s
            cnt[s] += 1
    best = max(cnt.values())
    return [s for s, c in cnt.items() if c == best]

for vals, expected in examples:
    got = freq_sum_iter(build_tree(vals))
    assert sorted(got) == expected
print("OK cách 3 trên ví dụ")

OK cách 3 trên ví dụ


In [13]:
class Solution:
    def findFrequentTreeSum(self, root: Optional[TreeNode]) -> List[int]:
        cnt = Counter()
        sums = {}
        stack = [(root, False)]
        while stack:
            node, done = stack.pop()
            if node is None:
                continue
            if not done:
                stack.append((node, True))
                stack.append((node.right, False))
                stack.append((node.left, False))
            else:
                s = node.val + sums.get(node.left, 0) + sums.get(node.right, 0)
                sums[node] = s
                cnt[s] += 1
        best = max(cnt.values())
        return [s for s, c in cnt.items() if c == best]

print([sorted(Solution().findFrequentTreeSum(build_tree(v))) for v, _ in examples])

[[-3, 2, 4], [2], [1]]


**Độ phức tạp (cách 3):** Time `O(n)`; Space `O(n)` (dict `sums` giữ mọi nút; ngăn xếp tối đa `O(n)`). Không còn rủi ro tràn ngăn xếp đệ quy.

### Cách 4 — Duyệt theo tầng rồi "cộng dồn ngược lên cha"

**Ý tưởng khác hẳn:** không dùng đệ quy cũng không dùng cờ hậu tự.

1. Duyệt cây **theo tầng** (BFS), ghi lại thứ tự thăm `order` và `parent[node]`.
2. Đặt `sums[node] = node.val` cho mọi nút.
3. Đi `order` **từ cuối về đầu** (tầng sâu trước): mỗi nút lúc này đã nhận đủ phần của các con (vì các con nằm sau nó trong `order`), nên `sums[node]` là tổng cây con cuối cùng → ghi `cnt`, rồi **cộng `sums[node]` vào cha**.

Vì sao đúng: trong thứ tự BFS, **con luôn đứng sau cha**; đi ngược thì **con luôn được xử lý trước cha**, đúng điều kiện hậu tự.

In [14]:
def freq_sum_bfs_parent(root):
    order, parent = [], {}
    queue = deque([root])
    parent[root] = None
    while queue:
        node = queue.popleft()
        order.append(node)
        for child in (node.left, node.right):
            if child is not None:
                parent[child] = node
                queue.append(child)

    sums = {node: node.val for node in order}
    cnt = Counter()
    for node in reversed(order):                # con trước cha
        cnt[sums[node]] += 1
        p = parent[node]
        if p is not None:
            sums[p] += sums[node]
    best = max(cnt.values())
    return [s for s, c in cnt.items() if c == best]

for vals, expected in examples:
    got = freq_sum_bfs_parent(build_tree(vals))
    assert sorted(got) == expected
print("OK cách 4 trên ví dụ")

OK cách 4 trên ví dụ


In [15]:
class Solution:
    def findFrequentTreeSum(self, root: Optional[TreeNode]) -> List[int]:
        order, parent = [], {root: None}
        queue = deque([root])
        while queue:
            node = queue.popleft()
            order.append(node)
            for child in (node.left, node.right):
                if child is not None:
                    parent[child] = node
                    queue.append(child)
        sums = {node: node.val for node in order}
        cnt = Counter()
        for node in reversed(order):
            cnt[sums[node]] += 1
            if parent[node] is not None:
                sums[parent[node]] += sums[node]
        best = max(cnt.values())
        return [s for s, c in cnt.items() if c == best]

print([sorted(Solution().findFrequentTreeSum(build_tree(v))) for v, _ in examples])

[[-3, 2, 4], [2], [1]]


**Độ phức tạp (cách 4):** Time `O(n)`; Space `O(n)` (`order`, `parent`, `sums`). Cách này **không đệ quy**, và không cần đánh dấu cờ — chỉ dựa vào tính chất "con đứng sau cha trong BFS".

### Cách 5 — Thay bảng băm bằng **sắp xếp** để đếm

**Ý tưởng:** nếu không muốn dùng `Counter`, ta gom **toàn bộ** các tổng vào một list rồi **sắp xếp**. Các giá trị bằng nhau sẽ đứng cạnh nhau thành từng đoạn; độ dài mỗi đoạn là tần suất. Dùng `itertools.groupby` để duyệt từng đoạn.

Với phần tính tổng ta dùng đệ quy hậu tự (đã quen) nhưng chỉ **append** vào list.

In [16]:
def freq_sum_sort(root):
    sums = []

    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        sums.append(s)
        return s

    dfs(root)
    sums.sort()
    groups = [(key, len(list(grp))) for key, grp in groupby(sums)]
    best = max(c for _, c in groups)
    return [key for key, c in groups if c == best]

for vals, expected in examples:
    got = freq_sum_sort(build_tree(vals))
    assert sorted(got) == expected
print("OK cách 5 trên ví dụ")
print(list(groupby([1, 1, 2, 2, 2, 5])) and [(k, len(list(g))) for k, g in groupby([1, 1, 2, 2, 2, 5])])

OK cách 5 trên ví dụ
[(1, 2), (2, 3), (5, 1)]


In [17]:
class Solution:
    def findFrequentTreeSum(self, root: Optional[TreeNode]) -> List[int]:
        sums = []

        def dfs(node: Optional[TreeNode]) -> int:
            if node is None:
                return 0
            s = node.val + dfs(node.left) + dfs(node.right)
            sums.append(s)
            return s

        dfs(root)
        sums.sort()
        groups = [(k, len(list(g))) for k, g in groupby(sums)]
        best = max(c for _, c in groups)
        return [k for k, c in groups if c == best]

print([sorted(Solution().findFrequentTreeSum(build_tree(v))) for v, _ in examples])

[[-3, 2, 4], [2], [1]]


**Độ phức tạp (cách 5):** Time `O(n log n)` do sắp xếp (vẫn nhanh với `n = 10^4`); Space `O(n)`. Ưu điểm: không phụ thuộc băm; kết quả tự trả về **theo thứ tự tăng dần** (hợp với đề vì thứ tự bất kỳ).

## 8. TRACE từng bước

Ta trace **cách 2** (DFS hậu tự + `Counter`) trên cây `[5, 2, -5]` bằng một hàm in ra mỗi lần **vào** và **ra** nút, với thụt đầu dòng theo độ sâu.

Bảng trạng thái:

| Sự kiện | Ý nghĩa |
|---|---|
| `vào x` | bắt đầu xử lý nút `x` (chưa biết tổng cây con) |
| `ra x: sL + sR + val = s` | đã biết tổng hai con, tính ra tổng cây con `s` |
| `cnt` | bảng đếm sau khi ghi nhận `s` |

In [18]:
def trace_dfs(root):
    cnt = Counter()

    def dfs(node, depth):
        pad = "  " * depth
        if node is None:
            return 0
        print(f"{pad}vào {node.val}")
        sl = dfs(node.left, depth + 1)
        sr = dfs(node.right, depth + 1)
        s = node.val + sl + sr
        cnt[s] += 1
        print(f"{pad}ra  {node.val}: {sl} + {sr} + {node.val} = {s}   cnt = {dict(cnt)}")
        return s

    dfs(root, 0)
    best = max(cnt.values())
    result = [s for s, c in cnt.items() if c == best]
    print("tần suất lớn nhất =", best, "-> kết quả", result)
    return result

trace_dfs(build_tree([5, 2, -5]))

vào 5
  vào 2
  ra  2: 0 + 0 + 2 = 2   cnt = {2: 1}
  vào -5
  ra  -5: 0 + 0 + -5 = -5   cnt = {2: 1, -5: 1}
ra  5: 2 + -5 + 5 = 2   cnt = {2: 2, -5: 1}
tần suất lớn nhất = 2 -> kết quả [2]


[2]

Quan sát: nút `2` ra trước nút `-5` (thứ tự trái -> phải), cả hai ghi `cnt` trước khi gốc `5` tính được tổng. Gốc tính `2 + (-5) + 5 = 2` và làm `cnt[2]` thành `2`. Bây giờ trace một cây sâu hơn có hòa nhau (để thấy "nhiều đáp án"): cây `[3, 1, 2, -1, -2, 1, None]`.

In [19]:
tree_tie = build_tree([3, 1, 2, -1, -2, 1])
show_levels(tree_tie)
res = trace_dfs(tree_tie)

tầng 0: [3]
tầng 1: [1, 2]
tầng 2: [-1, -2, 1, None]
vào 3
  vào 1
    vào -1
    ra  -1: 0 + 0 + -1 = -1   cnt = {-1: 1}
    vào -2
    ra  -2: 0 + 0 + -2 = -2   cnt = {-1: 1, -2: 1}
  ra  1: -1 + -2 + 1 = -2   cnt = {-1: 1, -2: 2}
  vào 2
    vào 1
    ra  1: 0 + 0 + 1 = 1   cnt = {-1: 1, -2: 2, 1: 1}
  ra  2: 1 + 0 + 2 = 3   cnt = {-1: 1, -2: 2, 1: 1, 3: 1}
ra  3: -2 + 3 + 3 = 4   cnt = {-1: 1, -2: 2, 1: 1, 3: 1, 4: 1}
tần suất lớn nhất = 2 -> kết quả [-2]


Ta cũng trace **cách 4** (cộng dồn ngược) để thấy `sums` thay đổi thế nào khi đi `order` từ cuối về đầu:

In [20]:
def trace_bfs_parent(root):
    order, parent = [], {root: None}
    queue = deque([root])
    while queue:
        node = queue.popleft()
        order.append(node)
        for child in (node.left, node.right):
            if child is not None:
                parent[child] = node
                queue.append(child)
    print("thứ tự BFS:", [x.val for x in order])
    sums = {node: node.val for node in order}
    cnt = Counter()
    for node in reversed(order):
        p = parent[node]
        print(f"xử lý {node.val:>3}: sum = {sums[node]:>3}", end="")
        cnt[sums[node]] += 1
        if p is not None:
            sums[p] += sums[node]
            print(f"  -> cộng vào cha {p.val} (sum cha = {sums[p]})")
        else:
            print("  (gốc)")
    print("cnt =", dict(cnt))

trace_bfs_parent(build_tree([5, 2, -5, 4, None, None, 1]))

thứ tự BFS: [5, 2, -5, 4, 1]
xử lý   1: sum =   1  -> cộng vào cha -5 (sum cha = -4)
xử lý   4: sum =   4  -> cộng vào cha 2 (sum cha = 6)
xử lý  -5: sum =  -4  -> cộng vào cha 5 (sum cha = 1)
xử lý   2: sum =   6  -> cộng vào cha 5 (sum cha = 7)
xử lý   5: sum =   7  (gốc)
cnt = {1: 1, 4: 1, -4: 1, 6: 1, 7: 1}


## 9. Code nộp LeetCode

Cách chuẩn: DFS hậu tự + `Counter`. Đúng class/method/signature của LeetCode (trên LeetCode `Counter`, `Optional`, `List`, `TreeNode` đã có sẵn; ở đây import để chạy được tại máy):

In [21]:
from collections import Counter
from typing import List, Optional


class Solution:
    def findFrequentTreeSum(self, root: Optional[TreeNode]) -> List[int]:
        cnt = Counter()

        def dfs(node: Optional[TreeNode]) -> int:
            if node is None:
                return 0
            s = node.val + dfs(node.left) + dfs(node.right)
            cnt[s] += 1
            return s

        dfs(root)
        best = max(cnt.values())
        return [s for s, c in cnt.items() if c == best]


for vals, expected in examples:
    got = Solution().findFrequentTreeSum(build_tree(vals))
    print(vals, "->", got)
    assert sorted(got) == expected

[5, 2, -3] -> [2, -3, 4]
[5, 2, -5] -> [2]
[1] -> [1]


## 10. Vì sao thuật toán đúng

**Hàm `dfs(node)` trả về đúng tổng cây con.** Quy nạp theo chiều cao: nút rỗng trả `0` (tổng rỗng). Với nút có con, theo giả thiết quy nạp `dfs(left)` và `dfs(right)` là tổng cây con của hai con; cộng với `node.val` ta được tổng của mọi nút trong cây con gốc `node` (mỗi nút thuộc đúng một trong ba phần: chính nó, cây con trái, cây con phải).

**`cnt` ghi đủ và không dư.** Mỗi nút của cây được `dfs` thăm **đúng một lần** (cây không có chu trình, mỗi nút có đúng một cha), và ngay khi thăm tính ra tổng của nó thì `cnt[s] += 1`. Vậy `cnt` là bảng tần suất của đúng `n` tổng cây con.

**Lọc kết quả đúng.** `best` là tần suất lớn nhất; các khóa có tần suất `best` chính là mọi tổng xuất hiện nhiều nhất. Trả về cả danh sách nên xử lý được trường hợp hòa. Vì `cnt` không rỗng (cây có ít nhất 1 nút), `max` không lỗi.

**Các cách khác** cho cùng tập tổng: cách 3 hậu tự bằng ngăn xếp (con luôn được tính xong trước khi nút được lấy ra lần hai), cách 4 dùng tính chất "con đứng sau cha trong BFS", cách 5 chỉ thay bước đếm bằng sắp xếp + gom nhóm.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mỗi nút tự tính lại | `O(n²)` (cây lệch) | `O(n)` |
| 2. DFS hậu tự + `Counter` | `O(n)` | `O(n)` |
| 3. Hậu tự không đệ quy | `O(n)` | `O(n)` |
| 4. BFS + cộng dồn lên cha | `O(n)` | `O(n)` |
| 5. Sắp xếp rồi gom nhóm | `O(n log n)` | `O(n)` |

Ở cách 2, ngăn xếp đệ quy cao `O(h)` với `h ≤ n`. Bảng `cnt` có tối đa `n` khóa. Mỗi tổng có thể đến `10^4 · 10^5 = 10^9` — vẫn là số nguyên Python bình thường, không lo tràn số.

## 12. Bộ test

### 12.1. Ví dụ đề, edge case và hàm kiểm tra hợp lệ

Vì có nhiều đáp án đúng, ta viết `is_valid(root_list, answer)` — **oracle độc lập**: tính mọi tổng cây con bằng cách làm khác hẳn (chuyển cây về list các nút, với mỗi nút duyệt riêng bằng ngăn xếp), rồi kiểm tra `answer` đúng là tập các tổng đạt tần suất lớn nhất, không trùng lặp.

In [22]:
def oracle_sums(root):
    """Độc lập với các cách ở trên: duyệt bằng ngăn xếp cho từng nút."""
    nodes, stack = [], [root]
    while stack:
        x = stack.pop()
        if x:
            nodes.append(x)
            stack += [x.left, x.right]
    out = []
    for x in nodes:
        total, st = 0, [x]
        while st:
            y = st.pop()
            if y:
                total += y.val
                st += [y.left, y.right]
        out.append(total)
    return out

def is_valid(root, answer):
    cnt = Counter(oracle_sums(root))
    best = max(cnt.values())
    expected = sorted(s for s, c in cnt.items() if c == best)
    return len(answer) == len(set(answer)) and sorted(answer) == expected

for vals, expected in examples:
    assert is_valid(build_tree(vals), expected)
print("is_valid chấp nhận các đáp án mẫu")
assert not is_valid(build_tree([5, 2, -5]), [2, -5])      # thừa
assert not is_valid(build_tree([5, 2, -3]), [2, 4])       # thiếu
assert not is_valid(build_tree([5, 2, -3]), [2, 2, 4, -3])  # lặp
print("is_valid từ chối đáp án sai")

is_valid chấp nhận các đáp án mẫu
is_valid từ chối đáp án sai


In [23]:
solvers = {
    "brute": freq_sum_brute,
    "dfs": freq_sum_dfs,
    "iter": freq_sum_iter,
    "bfs_parent": freq_sum_bfs_parent,
    "sort": freq_sum_sort,
    "leetcode": lambda r: Solution().findFrequentTreeSum(r),
}

edge_cases = [
    ([0], "một nút 0"),
    ([7], "một nút dương"),
    ([-4], "một nút âm"),
    ([1, 1, 1], "ba nút bằng nhau: tổng 1,1,3"),
    ([0, 0, 0, 0, 0], "toàn số 0: mọi tổng bằng 0"),
    ([1, -1], "tổng gốc = 0, con = -1"),
    ([1, None, 2, None, 3, None, 4], "cây lệch phải"),
    ([4, 3, None, 2, None, 1], "cây lệch trái"),
    ([1, 2, 3, 4, 5, 6, 7], "cây đầy đủ"),
    ([5, 2, -5], "ví dụ 2"),
    ([2, -2, -2], "gốc 2 con -2: tổng -2,-2,-2"),
    ([100000, -100000, 100000], "biên giá trị"),
]
for vals, note in edge_cases:
    for name, fn in solvers.items():
        got = fn(build_tree(vals))
        assert is_valid(build_tree(vals), got), (name, note, got)
    print("OK:", note)

OK: một nút 0
OK: một nút dương
OK: một nút âm
OK: ba nút bằng nhau: tổng 1,1,3
OK: toàn số 0: mọi tổng bằng 0
OK: tổng gốc = 0, con = -1
OK: cây lệch phải
OK: cây lệch trái
OK: cây đầy đủ
OK: ví dụ 2
OK: gốc 2 con -2: tổng -2,-2,-2
OK: biên giá trị


### 12.2. Phản ví dụ cho các cách làm sai

Ta viết vài bản sai điển hình và chứng minh `is_valid` bắt được chúng.

In [24]:
def wrong_no_val(root):
    """SAI: quên cộng node.val."""
    cnt = Counter()
    def dfs(node):
        if node is None:
            return 0
        s = dfs(node.left) + dfs(node.right)       # thiếu node.val
        cnt[s] += 1
        return s
    dfs(root)
    best = max(cnt.values())
    return [s for s, c in cnt.items() if c == best]

def wrong_single_answer(root):
    """SAI: chỉ trả về MỘT tổng, bỏ qua khi bị hòa."""
    cnt = Counter()
    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        cnt[s] += 1
        return s
    dfs(root)
    return [max(cnt, key=cnt.get)]

def wrong_root_only(root):
    """SAI: chỉ ghi tổng của gốc."""
    return [oracle_sums(root)[0]]

for fn in (wrong_no_val, wrong_single_answer, wrong_root_only):
    bad = [v for v, _ in examples if not is_valid(build_tree(v), fn(build_tree(v)))]
    print(fn.__name__, "sai trên các ví dụ:", bad)
    assert bad, fn.__name__

wrong_no_val sai trên các ví dụ: [[5, 2, -3], [5, 2, -5], [1]]
wrong_single_answer sai trên các ví dụ: [[5, 2, -3]]
wrong_root_only sai trên các ví dụ: [[5, 2, -3]]


### 12.3. Đối chiếu ngẫu nhiên (seed cố định) đúng ràng buộc

Ta sinh cây ngẫu nhiên `1 <= n <= 60`, giá trị trong khoảng nhỏ (để dễ có hòa) và cả khoảng lớn `[-10^5, 10^5]`, rồi kiểm tra **mọi cách** bằng `is_valid` và so các cách với nhau sau khi sắp xếp.

In [25]:
def random_tree_list(rng, n, lo, hi):
    """Sinh cây ngẫu nhiên n nút (n >= 1) với giá trị trong [lo, hi], trả về list kiểu LeetCode."""
    root = TreeNode(rng.randint(lo, hi))
    slots = [(root, "left"), (root, "right")]
    for _ in range(n - 1):
        k = rng.randrange(len(slots))
        slots[k], slots[-1] = slots[-1], slots[k]
        parent, side = slots.pop()
        child = TreeNode(rng.randint(lo, hi))
        setattr(parent, side, child)
        slots.append((child, "left"))
        slots.append((child, "right"))
    return tree_to_list(root)


rng = random.Random(508)
total = 0
for _ in range(1200):
    n = rng.randint(1, 60)
    lo, hi = rng.choice([(-2, 2), (-5, 5), (-100000, 100000)])
    vals = random_tree_list(rng, n, lo, hi)
    answers = {}
    for name, fn in solvers.items():
        root = build_tree(vals)
        got = fn(root)
        assert is_valid(root, got), (name, vals, got)
        answers[name] = sorted(got)
    assert len({tuple(v) for v in answers.values()}) == 1, (vals, answers)
    total += 1
print("đã đối chiếu", total, "cây ngẫu nhiên: 6 cách (kể cả code nộp) khớp và hợp lệ")

đã đối chiếu 1200 cây ngẫu nhiên: 6 cách (kể cả code nộp) khớp và hợp lệ


In [26]:
# Cây lệch 10^4 nút (tối đa theo ràng buộc): cách đệ quy cần nới giới hạn, cách lặp thì không
n = 10000
chain = [1]
for i in range(n - 1):
    chain += [None, 1]                 # mỗi nút chỉ có con phải
deep = build_tree(chain)
res_iter = freq_sum_iter(deep)
res_bfs = freq_sum_bfs_parent(deep)
old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(50000)
res_dfs = freq_sum_dfs(deep)
res_leet = Solution().findFrequentTreeSum(deep)
sys.setrecursionlimit(old_limit)
assert sorted(res_iter) == sorted(res_bfs) == sorted(res_dfs) == sorted(res_leet)
print("cây lệch 10^4 nút: tổng cây con là 1..10000, mỗi tổng 1 lần ->", len(res_iter), "đáp án")
assert len(res_iter) == n

cây lệch 10^4 nút: tổng cây con là 1..10000, mỗi tổng 1 lần -> 10000 đáp án


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên cộng `node.val`

**Code sai:**

In [27]:
class WrongSolution1:
    def findFrequentTreeSum(self, root):
        cnt = Counter()
        def dfs(node):
            if node is None:
                return 0
            s = dfs(node.left) + dfs(node.right)       # quên node.val
            cnt[s] += 1
            return s
        dfs(root)
        best = max(cnt.values())
        return [s for s, c in cnt.items() if c == best]

print("sai :", WrongSolution1().findFrequentTreeSum(build_tree([5, 2, -3])))
print("đúng:", Solution().findFrequentTreeSum(build_tree([5, 2, -3])))

sai : [0]
đúng: [2, -3, 4]


**Vì sao sai:** "tổng cây con" bao gồm **cả chính nút đó**. Với lá, thiếu `node.val` khiến mọi lá cho tổng `0`.

**Code đúng:** `s = node.val + dfs(node.left) + dfs(node.right)`.

### Lỗi 2: Chỉ trả về một đáp án khi bị hòa

In [28]:
class WrongSolution2:
    def findFrequentTreeSum(self, root):
        cnt = Counter()
        def dfs(node):
            if node is None:
                return 0
            s = node.val + dfs(node.left) + dfs(node.right)
            cnt[s] += 1
            return s
        dfs(root)
        return [max(cnt, key=cnt.get)]       # chỉ lấy 1 khóa

print("sai :", WrongSolution2().findFrequentTreeSum(build_tree([5, 2, -3])))
print("đúng:", sorted(Solution().findFrequentTreeSum(build_tree([5, 2, -3]))))

sai : [2]
đúng: [-3, 2, 4]


**Vì sao sai:** đề yêu cầu **tất cả** tổng có tần suất lớn nhất. Ví dụ 1 có ba tổng hòa nhau (mỗi tổng 1 lần), nhưng bản sai chỉ trả về một.

**Code đúng:** tính `best = max(cnt.values())` rồi lọc `[s for s, c in cnt.items() if c == best]`.

### Lỗi 3: Để trạng thái đếm lưu lại giữa các lần gọi

In [29]:
class WrongSolution3:
    cnt = Counter()                      # biến của LỚP: dùng chung giữa mọi lần gọi!

    def findFrequentTreeSum(self, root):
        def dfs(node):
            if node is None:
                return 0
            s = node.val + dfs(node.left) + dfs(node.right)
            WrongSolution3.cnt[s] += 1
            return s
        dfs(root)
        best = max(WrongSolution3.cnt.values())
        return [s for s, c in WrongSolution3.cnt.items() if c == best]

print("lần 1:", sorted(WrongSolution3().findFrequentTreeSum(build_tree([5, 2, -5]))))   # [2]
print("lần 2 (cây khác):", sorted(WrongSolution3().findFrequentTreeSum(build_tree([5, 2, -3]))))
print("đúng:", sorted(Solution().findFrequentTreeSum(build_tree([5, 2, -3]))))

lần 1: [2]
lần 2 (cây khác): [2]
đúng: [-3, 2, 4]


**Vì sao sai:** LeetCode chạy nhiều test case, thường **dùng lại cùng một tiến trình**. Nếu `cnt` là thuộc tính của lớp (hoặc biến toàn cục) mà không được đặt lại, kết quả của test trước còn dính sang test sau.

**Code đúng:** tạo `cnt = Counter()` **bên trong** `findFrequentTreeSum` (như mục 9), hoặc đặt lại ở đầu hàm.

### Lỗi 4: Nhầm "tần suất lớn nhất" với "tổng lớn nhất"

In [30]:
def wrong_largest_sum(root):
    sums = oracle_sums(root)
    return [max(sums)]                   # trả tổng LỚN NHẤT, không phải tổng XUẤT HIỆN NHIỀU NHẤT

t = build_tree([5, 2, -5])
print("sai :", wrong_largest_sum(t), "| đúng:", sorted(Solution().findFrequentTreeSum(t)))

sai : [2] | đúng: [2]


**Vì sao sai:** đề hỏi tổng xuất hiện **nhiều lần nhất**, không hỏi tổng lớn nhất. Ở `[5,2,-5]`, tổng lớn nhất là `2` (trùng ngẫu nhiên), nhưng với `[1, 5, 5]` thì tổng lớn nhất là `11` còn tổng nhiều nhất là `5`. Hãy chạy thử:

In [31]:
t = build_tree([1, 5, 5])
print("sai :", wrong_largest_sum(t), "| đúng:", sorted(Solution().findFrequentTreeSum(t)))
assert sorted(Solution().findFrequentTreeSum(t)) == [5]

sai : [11] | đúng: [5]


### Lỗi 5: Đệ quy quá sâu trên cây lệch (cục bộ)

Ở máy cá nhân, giới hạn đệ quy mặc định (~1000) có thể bị vượt với cây lệch `10^4` nút. Dùng cách 3 hoặc cách 4 (không đệ quy), hoặc tăng `sys.setrecursionlimit` (như test ở mục 12.3).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "tổng/số lượng/chiều cao **của cây con**" → nghĩ ngay tới **DFS hậu tự trả về giá trị** cho cha dùng lại. Đây là mẫu **Tree DP**.
- Hàm đệ quy vừa **trả về** một giá trị (tổng) cho cha, vừa **ghi chú vào một sổ** (`cnt`) → mẫu rất phổ biến (xem thêm: đường kính cây, đường đi có tổng lớn nhất, cây con có trung bình lớn nhất).
- Thấy "trả về tất cả giá trị có tần suất lớn nhất" → **đếm bằng `Counter`**, tính `max(values)` **một lần**, rồi lọc. Đừng dùng `max(cnt, key=cnt.get)` nếu có thể hòa.
- "Thứ tự bất kỳ" → khi tự test nhớ **sắp xếp** trước khi so sánh.
- Tránh biến cấp lớp/toàn cục lưu trạng thái; tạo mới trong hàm.
- Muốn tránh đệ quy sâu: ngăn xếp có cờ, hoặc BFS rồi đi ngược.

## 15. Biến thể

1. **Tổng cây con lớn nhất** (kiểu bài 1339/ 1373): thay `Counter` bằng `max`.
2. **Đếm số cây con có tổng bằng `target`:** vẫn một DFS; mỗi nút so `s == target`.
3. **Cây con có trung bình lớn nhất** (bài 1120): DFS trả về `(tổng, số nút)`.
4. **Tổng đường đi** thay vì tổng cây con (Path Sum III, bài 437): cần thêm tổng tiền tố.
5. **Top-k tổng xuất hiện nhiều nhất:** `cnt.most_common(k)`.
6. **Cây N-ary:** cùng ý tưởng, cộng qua danh sách `children`.

Ví dụ biến thể 2 và 3 bằng code:

In [32]:
def count_subtrees_with_sum(root, target):
    count = 0
    def dfs(node):
        nonlocal count
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        if s == target:
            count += 1
        return s
    dfs(root)
    return count

def max_average_subtree(root):
    best = 0.0
    def dfs(node):
        nonlocal best
        if node is None:
            return 0, 0
        ls, lc = dfs(node.left)
        rs, rc = dfs(node.right)
        s, c = ls + rs + node.val, lc + rc + 1
        best = max(best, s / c) if c else best
        return s, c
    dfs(root)
    return best

t = build_tree([5, 2, -5])
print(count_subtrees_with_sum(t, 2), "(tổng 2 xuất hiện 2 lần)")
print(max_average_subtree(build_tree([5, 6, 1])))      # 6.0 (nút 6)

2 (tổng 2 xuất hiện 2 lần)
6.0


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tự tính lại từng nút | Mỗi nút một lượt duyệt riêng | `O(n²)` | `O(n)` | Chỉ để kiểm chứng/đối chiếu |
| 2. DFS hậu tự + Counter | Trả tổng cho cha, ghi `cnt` | `O(n)` | `O(n)` | **Cách chuẩn để nộp** |
| 3. Hậu tự bằng ngăn xếp | Cờ "đã xử lý con" | `O(n)` | `O(n)` | Cây rất sâu, tránh `RecursionError` |
| 4. BFS + cộng dồn lên cha | Đi ngược thứ tự BFS | `O(n)` | `O(n)` | Khi muốn code không đệ quy, không cờ |
| 5. Sắp xếp + `groupby` | Đếm bằng gom nhóm sau khi sort | `O(n log n)` | `O(n)` | Khi không muốn dùng băm, cần kết quả có thứ tự |

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán trước khi chạy: `Counter(oracle_sums(build_tree([1, 2, 3])))` ra gì? (Gợi ý: thứ tự các khóa có thể khác; bạn chỉ cần đúng nội dung.)

In [33]:
print(Counter(oracle_sums(build_tree([1, 2, 3]))))

Counter({6: 1, 3: 1, 2: 1})


**Bài 2 (tính tay).** Với cây `[1, 2, 3, 4]` (nút 4 là con trái của 2), tìm tổng cây con của cả 4 nút rồi suy ra đáp án bài toán (sắp xếp tăng dần). Gán vào `ans2` (đang là `None`).

In [34]:
ans2 = None      # TODO: list đã sắp xếp tăng dần

**Bài 3 (điền TODO).** Viết hàm `collect_sums(root)` trả về **list** mọi tổng cây con theo thứ tự hậu tự (con trái, con phải, rồi nút). Ví dụ `[5,2,-3]` -> `[2, -3, 4]`.

In [35]:
def collect_sums(root):
    # TODO: dùng một hàm dfs hậu tự trả về tổng, append vào list
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_find_frequent(root)` theo cách 2 (DFS hậu tự + `Counter`) và trả về list các tổng xuất hiện nhiều nhất.

In [36]:
def my_find_frequent(root):
    # TODO
    pass

**Bài 5 (mini challenge).** Viết `second_most_frequent(root)`: trả về list mọi tổng có tần suất **lớn thứ hai** (giá trị tần suất khác nhau, nếu không có tần suất thứ hai thì trả `[]`). Ví dụ `[5,2,-5]`: tần suất là `{2: 2, -5: 1}` -> tần suất lớn thứ hai là `1` -> `[-5]`.

In [37]:
def second_most_frequent(root):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: tính tổng của từng nút: hai lá và gốc.
- Bài 2: tính từ dưới lên: nút 4, rồi nút 2, nút 3, gốc.
- Bài 3: hàm đệ quy trả về tổng; mỗi lần tính xong thì `append`.
- Bài 4: chép lại khung: `cnt`, `dfs`, `best`, lọc.
- Bài 5: lấy tập các tần suất khác nhau.

**Mức 2 (vừa):**
- Bài 2: `sum(4)=4`, `sum(2)=2+4`, `sum(3)=3`, `sum(1)=1+6+3`; có tổng nào lặp không?
- Bài 3: `def dfs(node): if not node: return 0; s = ...; res.append(s); return s`.
- Bài 5: `freqs = sorted(set(cnt.values()), reverse=True)`; nếu `len(freqs) < 2` thì `[]`.

**Mức 3 (gần đáp án):**
- Bài 3: `res = []` ngoài `dfs`, `dfs(root)`, `return res`.
- Bài 4: xem lại mục 9.
- Bài 5: `target = freqs[1]`, trả `[s for s, c in cnt.items() if c == target]`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [38]:
# Bài 1: lá 2, lá 3, gốc 1 + 2 + 3 = 6 => {2: 1, 3: 1, 6: 1}
c1 = Counter(oracle_sums(build_tree([1, 2, 3])))
assert c1 == Counter({2: 1, 3: 1, 6: 1})

# Bài 2: sums 4, 6, 3, 10 đều khác nhau, mỗi tổng một lần => tất cả
ans2 = [3, 4, 6, 10]
assert ans2 == sorted(Solution().findFrequentTreeSum(build_tree([1, 2, 3, 4])))
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [39]:
# Bài 3
def collect_sums(root):
    res = []
    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        res.append(s)
        return s
    dfs(root)
    return res

assert collect_sums(build_tree([5, 2, -3])) == [2, -3, 4]
assert collect_sums(build_tree([1])) == [1]
assert sorted(collect_sums(build_tree([5, 2, -5]))) == sorted(oracle_sums(build_tree([5, 2, -5])))
print("Bài 3 OK")

Bài 3 OK


In [40]:
# Bài 4
def my_find_frequent(root):
    cnt = Counter()
    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        cnt[s] += 1
        return s
    dfs(root)
    best = max(cnt.values())
    return [s for s, c in cnt.items() if c == best]

rng = random.Random(1)
for _ in range(200):
    vals = random_tree_list(rng, rng.randint(1, 30), -3, 3)
    root = build_tree(vals)
    assert is_valid(root, my_find_frequent(root))
for v, e in examples:
    assert sorted(my_find_frequent(build_tree(v))) == e
print("Bài 4 OK")

Bài 4 OK


In [41]:
# Bài 5
def second_most_frequent(root):
    cnt = Counter(oracle_sums(root))
    freqs = sorted(set(cnt.values()), reverse=True)
    if len(freqs) < 2:
        return []
    target = freqs[1]
    return [s for s, c in cnt.items() if c == target]

assert second_most_frequent(build_tree([5, 2, -5])) == [-5]
assert second_most_frequent(build_tree([5, 2, -3])) == []            # mọi tần suất bằng nhau
t5 = build_tree([0, 1, 1, 1, 1, 2])
c5 = Counter(oracle_sums(t5))
f5 = sorted(set(c5.values()), reverse=True)
exp5 = sorted(s for s, c in c5.items() if c == f5[1]) if len(f5) > 1 else []
assert sorted(second_most_frequent(t5)) == exp5
print("tần suất", dict(c5), "-> lớn thứ hai:", exp5)
print("Bài 5 OK")

tần suất {6: 1, 3: 2, 2: 1, 1: 2} -> lớn thứ hai: [2, 6]
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "trả về tổng cho cha, ghi chú vào sổ":**

```python
class Solution:
    def findFrequentTreeSum(self, root):
        cnt = Counter()                       # tạo MỚI trong hàm

        def dfs(node):
            if node is None:
                return 0
            s = node.val + dfs(node.left) + dfs(node.right)
            cnt[s] += 1                       # ghi tổng của nút này
            return s                          # trả về cho cha

        dfs(root)
        best = max(cnt.values())
        return [s for s, c in cnt.items() if c == best]
```

**Checklist trước khi submit:**

- [ ] `s = node.val + dfs(left) + dfs(right)` — có cộng `node.val`?
- [ ] Nút rỗng trả về `0` và **không** được ghi vào `cnt`?
- [ ] `cnt[s] += 1` thực hiện **cho mọi nút** (cả lá, cả gốc)?
- [ ] Trả về **tất cả** khóa có tần suất `best` (không chỉ một)?
- [ ] `Counter` tạo bên trong hàm, không dùng biến lớp/toàn cục lưu lại giữa các test?
- [ ] Đã thử: một nút, cây lệch, toàn số `0`, giá trị âm?
- [ ] Khi test tự viết: so sánh sau khi `sorted` hoặc dùng hàm kiểm tra hợp lệ?